In [1]:
# cell 1
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Using Apple MPS (Metal Performance Shaders) Acceleration")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("✅ Using NVIDIA CUDA Acceleration")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Using CPU (Might be slow)")

# path config
if os.path.exists('/kaggle/input'):
    DATA_PATH = '/kaggle/input/trademaster-cup-2025/' 
else:
    DATA_PATH = '../data/raw' 

# Hyperparameters
BATCH_SIZE = 2048  # Larger batch size for faster training on GPU/MPS
LEARNING_RATE = 0.001
EPOCHS = 20

✅ Using Apple MPS (Metal Performance Shaders) Acceleration


In [2]:
# %% Cell 1.5: The Global Synchronization Signal
import pandas as pd

# Load train data to get exact length
df_train_full = pd.read_csv('../data/raw/train_v2.csv')

# GLOBAL SPLIT INDEX (90% Train, 10% Validation)
# All models MUST use this variable.
SPLIT_IDX = int(len(df_train_full) * 0.90)
VAL_IDS = df_train_full['id'].iloc[SPLIT_IDX:].values

print(f"🔒 Global Split Index: {SPLIT_IDX}")
print(f"🔒 Validation Rows: {len(VAL_IDS)}")
# Expecting ~13,940 rows based on your logs

🔒 Global Split Index: 125452
🔒 Validation Rows: 13940


In [3]:
# CELL 2 & 3: Feature Engineering (Optimized & Safe)
print("🚀 Starting Sniper Data Pipeline (Safe Mode)...")

# 1. Load Data
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_v2.csv')).sort_values(['date_id', 'minute_id'])
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_v2.csv')).sort_values(['date_id', 'minute_id'])

# 2. Define Features
target_cols = ['target_short', 'target_medium', 'target_long']
ignore = ['id', 'date_id', 'minute_id'] + target_cols
base_features = [c for c in train_df.columns if c not in ignore]
vip_features = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']

# --- OPTIMIZED FUNCTIONS (No Fragmentation) ---
def create_lags_fast(df, features, lags=[1, 2, 3, 5]):
    print(f"   Generating Lags...")
    new_cols = []
    for col in features:
        for lag in lags:
            s = df.groupby('date_id')[col].shift(lag)
            s.name = f'{col}_lag{lag}'
            new_cols.append(s)
    return pd.concat([df] + new_cols, axis=1)

def create_vip_features(df):
    print(f"   Generating VIP Interactions & Rolling Stats...")
    new_cols = []
    
    # Rolling Stats for VIPs
    windows = [5, 10, 20]
    for col in vip_features:
        for w in windows:
            # Mean
            s_mean = df.groupby('date_id')[col].transform(lambda x: x.rolling(w).mean())
            s_mean.name = f'{col}_mean_{w}'
            new_cols.append(s_mean)
            # Std
            s_std = df.groupby('date_id')[col].transform(lambda x: x.rolling(w).std())
            s_std.name = f'{col}_std_{w}'
            new_cols.append(s_std)

    # Interactions with Feature 19 (The King)
    f19 = df['feature_19']
    for col in ['feature_5', 'feature_27']:
        s_mult = f19 * df[col]
        s_mult.name = f'feat_19_x_{col}'
        new_cols.append(s_mult)
        
    return pd.concat([df] + new_cols, axis=1)

# 3. Apply Engineering
train_eng = create_vip_features(create_lags_fast(train_df, base_features))
test_eng = create_vip_features(create_lags_fast(test_df, base_features))

# 4. Prepare Arrays
final_features = [c for c in train_eng.columns if c not in ignore]
print(f"✅ Final Feature Count: {len(final_features)}")

X = train_eng[final_features].astype(np.float32).values
y = train_eng[target_cols].astype(np.float32).values
X_test = test_eng[final_features].astype(np.float32).values

# Clean NaNs
X = np.nan_to_num(X, nan=0.0)
X_test = np.nan_to_num(X_test, nan=0.0)

# --- STRICT SCALING FIX ---
# We define the split FIRST, then scale based ONLY on Train.
split_idx = int(len(X) * 0.90)

X_tr_raw = X[:split_idx]
X_val_raw = X[split_idx:]
y_tr = y[:split_idx]
y_val = y[split_idx:]

print("   Fitting Scaler on TRAIN only...")
scaler = RobustScaler()
X_tr_scaled = scaler.fit_transform(X_tr_raw) # Fit on Train
X_val_scaled = scaler.transform(X_val_raw)   # Apply to Val
X_test_scaled = scaler.transform(X_test)     # Apply to Test

print("✅ Data Ready & Leak-Free.")

🚀 Starting Sniper Data Pipeline (Safe Mode)...
   Generating Lags...
   Generating VIP Interactions & Rolling Stats...
   Generating Lags...
   Generating VIP Interactions & Rolling Stats...
✅ Final Feature Count: 187
   Fitting Scaler on TRAIN only...
✅ Data Ready & Leak-Free.


/Users/baistan/Developer/active/kaggle-trademaster-hkust/venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1767: RuntimeWarning: overflow encountered in divide
  X /= self.scale_


In [4]:
# %% Cell 3b: Feature Injection (Ranks & Deltas) - LEAK FREE
import numpy as np
import pandas as pd

print("💉 Injecting Relative Strength & Delta Features (Safe Mode)...")

def create_rank_features(df, vip_cols):
    df_eng = df.copy()
    print(f"   Generating Rolling Ranks for {len(vip_cols)} VIPs...")
    for col in vip_cols:
        # "Is the current value high compared to the last hour?"
        df_eng[f'{col}_rank'] = df_eng.groupby('date_id')[col].transform(
            lambda x: x.rolling(window=60, min_periods=10).rank(pct=True)
        )
    return df_eng

def create_delta_features(df, vip_cols):
    df_eng = df.copy()
    print("   Generating Velocity Deltas...")
    for col in vip_cols:
        if f'{col}_lag1' in df_eng.columns:
            # Current - Yesterday
            df_eng[f'{col}_delta'] = df_eng[col] - df_eng[f'{col}_lag1']
    return df_eng

# 1. APPLY TO TRAIN
vips = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']
train_eng = create_rank_features(train_eng, vips)
train_eng = create_delta_features(train_eng, vips)

# 2. APPLY TO TEST
test_eng = create_rank_features(test_eng, vips)
test_eng = create_delta_features(test_eng, vips)

# 3. UPDATE X ARRAYS (RAW)
# We update the raw 'X' matrix but DO NOT scale it yet.
new_features = [c for c in train_eng.columns if 'rank' in c or 'delta' in c]
print(f"   ✅ Added {len(new_features)} New Features (Ranks + Deltas)")

ignore_cols = ['id', 'date_id', 'minute_id', 'target_short', 'target_medium', 'target_long', 'row_id']
final_cols = [c for c in train_eng.columns if c not in ignore_cols]
print(f"   Total Feature Count: {len(final_cols)}")

X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

# Light sanitization (Nan to 0) to keep raw data clean
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

print("💉 Injection Complete (Raw Features Updated).")

💉 Injecting Relative Strength & Delta Features (Safe Mode)...
   Generating Rolling Ranks for 5 VIPs...
   Generating Velocity Deltas...
   Generating Rolling Ranks for 5 VIPs...
   Generating Velocity Deltas...
   ✅ Added 10 New Features (Ranks + Deltas)
   Total Feature Count: 197
💉 Injection Complete (Raw Features Updated).


In [5]:
# %% Cell 3c: Feature Injection (Market Context) - LEAK FREE
print("💉 Injecting Market Context & Divergence Features (Safe Mode)...")

def create_market_features(df, vip_cols):
    df_eng = df.copy()
    print(f"   Generating Global Context for {len(vip_cols)} VIPs...")
    for col in vip_cols:
        # 1. EXPANDING MEAN (The "Mood" SO FAR)
        market_mean = df_eng.groupby('date_id')[col].transform(lambda x: x.expanding().mean())
        df_eng[f'global_mean_{col}'] = market_mean
        
        # 2. DIVERGENCE (The "Alpha")
        df_eng[f'divergence_{col}'] = df_eng[col] - market_mean
        
        # 3. EXPANDING STD (The "Panic")
        df_eng[f'global_std_{col}'] = df_eng.groupby('date_id')[col].transform(lambda x: x.expanding().std())
    return df_eng

# 1. APPLY
vips_context = ['feature_19', 'feature_5', 'feature_27']
train_eng = create_market_features(train_eng, vips_context)
test_eng = create_market_features(test_eng, vips_context)

# 2. UPDATE X ARRAYS (RAW)
new_features = [c for c in train_eng.columns if 'global' in c or 'divergence' in c]
print(f"   ✅ Added {len(new_features)} Market Context Features")

ignore_cols = ['id', 'date_id', 'minute_id', 'target_short', 'target_medium', 'target_long', 'row_id']
final_cols = [c for c in train_eng.columns if c not in ignore_cols]
print(f"   Total Feature Count: {len(final_cols)}")

X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

print("💉 Market Context Injection Complete (Raw Features Updated).")

💉 Injecting Market Context & Divergence Features (Safe Mode)...
   Generating Global Context for 3 VIPs...
   Generating Global Context for 3 VIPs...
   ✅ Added 9 Market Context Features
   Total Feature Count: 206
💉 Market Context Injection Complete (Raw Features Updated).


In [6]:
# %% Cell 3d: Single-Stock Special - LEAK FREE
print("💉 Injecting Single-Stock Features (Accumulators & Clock)...")

def create_intraday_features(df, vip_cols):
    df_eng = df.copy()
    
    # 1. THE CLOCK
    print("   Generating Clock Features...")
    df_eng['dist_from_open'] = df_eng['minute_id']
    max_min = df_eng['minute_id'].max()
    df_eng['dist_from_close'] = max_min - df_eng['minute_id']
    
    # 2. PSEUDO-PRICE
    print(f"   Generating Pseudo-Prices for {len(vip_cols)} VIPs...")
    for col in vip_cols:
        # Cumulative Sum
        df_eng[f'cum_{col}'] = df_eng.groupby('date_id')[col].cumsum()
        
        # Breakout Features
        day_max = df_eng.groupby('date_id')[col].cummax()
        day_min = df_eng.groupby('date_id')[col].cummin()
        range_vals = day_max - day_min
        range_vals = np.where(range_vals == 0, 1, range_vals)
        df_eng[f'day_position_{col}'] = (df_eng[col] - day_min) / range_vals

    return df_eng

# 1. APPLY
vips_single = ['feature_19', 'feature_5', 'feature_27']
train_eng = create_intraday_features(train_eng, vips_single)
test_eng = create_intraday_features(test_eng, vips_single)

# 2. UPDATE X ARRAYS (RAW)
ignore_cols = ['id', 'date_id', 'minute_id', 'target_short', 'target_medium', 'target_long', 'row_id']
final_cols = [c for c in train_eng.columns if c not in ignore_cols]
print(f"   Total Feature Count: {len(final_cols)}")

X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

# 3. FINAL RAW SANITIZATION
# We leave 'X' unscaled. Scaling happens in Cell 7.
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

print("💉 Single-Stock Injection Complete. READY FOR SCALING (Cell 7).")

💉 Injecting Single-Stock Features (Accumulators & Clock)...
   Generating Clock Features...
   Generating Pseudo-Prices for 3 VIPs...
   Generating Clock Features...
   Generating Pseudo-Prices for 3 VIPs...
   Total Feature Count: 214
💉 Single-Stock Injection Complete. READY FOR SCALING (Cell 7).


In [7]:
# %% CELL 7 (FIXED): RETROACTIVE LEAK REPAIR
# This replaces the leaked scaling from Cells 4/5/6 with a STRICT split.

print("🔧 REPAIRING PIPELINE (Removing Scaling Leakage)...")

from sklearn.preprocessing import RobustScaler
import numpy as np

# 1. RETRIEVE RAW DATA
# We rely on 'X' (from the end of Cell 6), which holds the raw feature values.
if 'X' not in locals():
    raise ValueError("⚠️ Raw 'X' missing. Please run Cells 1-6 first.")

# 2. STRICT SPLIT (Split BEFORE Scaling)
split_idx = int(len(X) * 0.90)

print(f"   Splitting {len(X)} rows at index {split_idx}...")
X_tr_raw = X[:split_idx]
X_val_raw = X[split_idx:]

# X_test is already separate, but we sanitize it just in case
if 'X_test' not in locals():
    raise ValueError("⚠️ X_test missing.")
X_test_raw = X_test

# 3. STRICT SCALING (Fit on Train ONLY)
print("   🛡️ Fitting RobustScaler on TRAIN set only...")
scaler = RobustScaler()

# FIT on Train, TRANSFORM Train
X_tr_scaled = scaler.fit_transform(X_tr_raw)

# TRANSFORM Validation & Test (Using Train stats)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 4. FINAL SANITIZATION
# Neural Networks & K-Means hate NaNs/Infs
X_tr_scaled = np.nan_to_num(X_tr_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_val_scaled = np.nan_to_num(X_val_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

print(f"✅ LEAKAGE REMOVED.")
print(f"   X_tr_scaled shape: {X_tr_scaled.shape}")
print(f"   X_val_scaled shape: {X_val_scaled.shape}")
print("   (You can now proceed to Cell 8 for Clustering)")

🔧 REPAIRING PIPELINE (Removing Scaling Leakage)...
   Splitting 139392 rows at index 125452...
   🛡️ Fitting RobustScaler on TRAIN set only...
✅ LEAKAGE REMOVED.
   X_tr_scaled shape: (125452, 214)
   X_val_scaled shape: (13940, 214)
   (You can now proceed to Cell 8 for Clustering)


In [8]:
# CELL 8: Cluster Features (Honest Mode)
from sklearn.cluster import KMeans

print("🚀 Generating Cluster Features (Honest Mode)...")

# 1. Fit on TRAIN Only
kmeans = KMeans(n_clusters=7, random_state=SEED, n_init=10)
# Subsample for speed, but strictly from train
kmeans.fit(X_tr_scaled[::10]) 

# 2. Predict All
tr_clusters = kmeans.predict(X_tr_scaled)
val_clusters = kmeans.predict(X_val_scaled)
test_clusters = kmeans.predict(X_test_scaled)

# 3. One-Hot Encode & Attach
def add_clusters(X_data, clusters):
    # Create one-hot
    oh = np.eye(7)[clusters]
    return np.hstack([X_data, oh])

X_tr_final = add_clusters(X_tr_scaled, tr_clusters)
X_val_final = add_clusters(X_val_scaled, val_clusters)
X_test_final = add_clusters(X_test_scaled, test_clusters)

print(f"✅ Clusters Added. New Shape: {X_tr_final.shape}")

🚀 Generating Cluster Features (Honest Mode)...
✅ Clusters Added. New Shape: (125452, 221)


In [9]:
# %% CELL 9: SAVE PROCESSED DATA (Dual Format - Corrected)
import os
import numpy as np

# Create folder if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

print("💾 Saving arrays for Model Ensembling...")

# 1. SAVE CLEAN SCALED DATA (Best for MLP / Neural Networks)
# These come from Cell 7 (The Leak-Free Repair Cell)
if 'X_tr_scaled' in locals():
    np.save('../data/processed/X_tr_scaled.npy', X_tr_scaled)
    np.save('../data/processed/X_val_scaled.npy', X_val_scaled)
    np.save('../data/processed/X_test_scaled.npy', X_test_scaled)
    print("   ✅ Saved Clean Scaled Data (for MLP).")
else:
    print("   ⚠️ WARNING: X_tr_scaled not found. Did you run Cell 7?")

# 2. SAVE CLUSTERED DATA (Best for Trees: XGB/LGB/Cat)
# These come from Cell 8
if 'X_tr_final' in locals():
    np.save('../data/processed/X_tr_final.npy', X_tr_final)
    np.save('../data/processed/X_val_final.npy', X_val_final)
    np.save('../data/processed/X_test_final.npy', X_test_final)
    print("   ✅ Saved Clustered Data (for Trees).")
else:
    print("   ⚠️ WARNING: X_tr_final not found. Did you run Cell 8?")

# 3. SAVE TARGETS & IDs
np.save('../data/processed/y_tr.npy', y_tr)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/test_ids.npy', test_df['id'].values)

print("✅ All Data saved to ../data/processed/")

💾 Saving arrays for Model Ensembling...
   ✅ Saved Clean Scaled Data (for MLP).
   ✅ Saved Clustered Data (for Trees).
✅ All Data saved to ../data/processed/


In [10]:
# %% CELL 10: XGBoost Purist Chain - CORRECTED
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

print("🔗 INITIALIZING PURIST XGBOOST CHAIN...")

if 'X_tr_final' not in locals():
    raise ValueError("⚠️ Please run Cell 8 (Clustering) first.")

# 1. Reset Feature Sets
X_tr_curr = X_tr_final.copy()
X_val_curr = X_val_final.copy()
X_test_curr = X_test_final.copy()

params = {
    'objective': 'reg:absoluteerror',
    'eval_metric': 'mae',
    'tree_method': 'hist',
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'n_jobs': -1,
    'verbosity': 0
}

xgb_preds = np.zeros((len(X_test_curr), 3))

for i in range(3): 
    print(f"\n🔗 Link {i+1}/3: Target {i}")
    
    # --- A. GENERATE HONEST OOF PREDS (For Next Link) ---
    honest_feat = None
    if i < 2:
        print(f"      ⚙️ Generating Honest History Features...")
        honest_feat = np.zeros(len(X_tr_curr))
        tscv = TimeSeriesSplit(n_splits=5)
        
        for train_idx, val_idx in tscv.split(X_tr_curr):
            d_past = xgb.DMatrix(X_tr_curr[train_idx], label=y_tr[train_idx, i])
            d_future = xgb.DMatrix(X_tr_curr[val_idx])
            
            model_inner = xgb.train(params, d_past, num_boost_round=300, verbose_eval=False)
            honest_feat[val_idx] = model_inner.predict(d_future)
        
    # --- B. TRAIN MAIN MODEL (On Current Features) ---
    print(f"      Training Main Model...")
    dtrain = xgb.DMatrix(X_tr_curr, label=y_tr[:, i])
    dval = xgb.DMatrix(X_val_curr, label=y_val[:, i])
    
    model = xgb.train(params, dtrain, num_boost_round=2000, 
                      evals=[(dval, 'valid')], early_stopping_rounds=50, verbose_eval=False)
    
    print(f"      ✅ Score: {model.best_score:.6f}")
    
    # Predict
    val_pred = model.predict(dval)
    test_pred = model.predict(xgb.DMatrix(X_test_curr))
    xgb_preds[:, i] = test_pred
    
    # --- C. UPDATE FEATURES FOR NEXT LINK ---
    if i < 2:
        print(f"      🔗 Chaining predictions to next target...")
        X_tr_curr = np.hstack([X_tr_curr, honest_feat.reshape(-1, 1)])
        X_val_curr = np.hstack([X_val_curr, val_pred.reshape(-1, 1)])
        X_test_curr = np.hstack([X_test_curr, test_pred.reshape(-1, 1)])

# Save
sub_xgb = pd.DataFrame({'id': test_df['id']})
sub_xgb[['target_short', 'target_medium', 'target_long']] = xgb_preds
for c in target_cols: sub_xgb[c] -= sub_xgb[c].mean()
sub_xgb.to_csv('submission_xgb_purist.csv', index=False)
print("🚀 Saved Purist XGB.")

🔗 INITIALIZING PURIST XGBOOST CHAIN...

🔗 Link 1/3: Target 0
      ⚙️ Generating Honest History Features...
      Training Main Model...
      ✅ Score: 0.002861
      🔗 Chaining predictions to next target...

🔗 Link 2/3: Target 1
      ⚙️ Generating Honest History Features...
      Training Main Model...
      ✅ Score: 0.007274
      🔗 Chaining predictions to next target...

🔗 Link 3/3: Target 2
      Training Main Model...
      ✅ Score: 0.015399
🚀 Saved Purist XGB.


In [11]:
# %% CELL 13: LightGBM Purist Chain (Strict Walk-Forward) - CORRECTED
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

print("🔗 INITIALIZING PURIST LIGHTGBM CHAIN...")

# 1. Reset Feature Sets
if 'X_tr_final' not in locals():
    raise ValueError("⚠️ Please run Cell 8 (Clustering) first.")

X_tr_curr = X_tr_final.copy()
X_val_curr = X_val_final.copy()
X_test_curr = X_test_final.copy()

# 2. Params
params_lgb = {
    'objective': 'regression_l1', 
    'metric': 'mae', 
    'boosting_type': 'gbdt',
    'n_estimators': 2000, 
    'learning_rate': 0.03, 
    'num_leaves': 31,
    'max_depth': 6, 
    'feature_fraction': 0.7, 
    'bagging_fraction': 0.7,
    'verbosity': -1, 
    'n_jobs': -1
}

lgb_preds = np.zeros((len(X_test_curr), 3))

for i in range(3):
    print(f"\n🔗 Link {i+1}/3: Target {i}")
    
    # --- A. GENERATE HONEST OOF PREDS (For Next Link) ---
    # We calculate this NOW, but attach it LATER.
    honest_feat = None
    if i < 2:
        print(f"      ⚙️ Generating Honest History Features...")
        honest_feat = np.zeros(len(X_tr_curr))
        tscv = TimeSeriesSplit(n_splits=5)
        
        for t_idx, v_idx in tscv.split(X_tr_curr):
            d_inner_tr = lgb.Dataset(X_tr_curr[t_idx], label=y_tr[t_idx, i])
            d_inner_val = lgb.Dataset(X_tr_curr[v_idx], label=y_tr[v_idx, i], reference=d_inner_tr)
            
            model_inner = lgb.train(params_lgb, d_inner_tr, valid_sets=[d_inner_val],
                                    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
            
            honest_feat[v_idx] = model_inner.predict(X_tr_curr[v_idx])

    # --- B. TRAIN MAIN MODEL (On Current Features) ---
    print(f"      Training Main Model...")
    # NOTE: X_tr_curr and X_val_curr have the SAME number of columns here.
    dtrain = lgb.Dataset(X_tr_curr, label=y_tr[:, i])
    dval = lgb.Dataset(X_val_curr, label=y_val[:, i], reference=dtrain)
    
    model = lgb.train(params_lgb, dtrain, valid_sets=[dval], 
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    # Predict (This will now work because shapes match)
    val_pred = model.predict(X_val_curr)
    test_pred = model.predict(X_test_curr)
    lgb_preds[:, i] = test_pred
    
    # --- C. UPDATE FEATURES FOR NEXT LINK ---
    if i < 2:
        # NOW we append the features to ALL sets simultaneously
        print(f"      🔗 Chaining predictions to next target...")
        X_tr_curr = np.hstack([X_tr_curr, honest_feat.reshape(-1, 1)])
        X_val_curr = np.hstack([X_val_curr, val_pred.reshape(-1, 1)])
        X_test_curr = np.hstack([X_test_curr, test_pred.reshape(-1, 1)])

# Save
sub_lgb = pd.DataFrame({'id': test_df['id']})
sub_lgb[['target_short', 'target_medium', 'target_long']] = lgb_preds
for c in target_cols: sub_lgb[c] -= sub_lgb[c].mean()
sub_lgb.to_csv('submission_lgbm_purist.csv', index=False)
print("🚀 Saved Purist LightGBM.")

🔗 INITIALIZING PURIST LIGHTGBM CHAIN...

🔗 Link 1/3: Target 0
      ⚙️ Generating Honest History Features...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 0.00392523
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 0.00376384
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's l1: 0.00431604
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[15]	valid_0's l1: 0.00386464
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 0.00315041
      Training Main Model...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[24]	valid_0's l1: 0.0028581
      🔗 Chaining predictions to next target...

🔗 Link 2/3: Target 1
      ⚙️ Generating Honest History Featur

In [12]:
# %% CELL 14: CatBoost Purist Chain - CORRECTED
from catboost import CatBoostRegressor, Pool
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

print("🔗 INITIALIZING PURIST CATBOOST CHAIN...")

if 'X_tr_final' not in locals():
    raise ValueError("⚠️ Please run Cell 8 (Clustering) first.")

# 1. Reset Feature Sets
X_tr_curr = X_tr_final.copy()
X_val_curr = X_val_final.copy()
X_test_curr = X_test_final.copy()

params_cat = {
    'loss_function': 'MAE', 
    'eval_metric': 'MAE', 
    'iterations': 1500,
    'learning_rate': 0.05, 
    'depth': 6, 
    'l2_leaf_reg': 3,
    'verbose': 0, 
    'task_type': 'CPU', 
    'allow_writing_files': False
}

cat_preds = np.zeros((len(X_test_curr), 3))

for i in range(3):
    print(f"\n🔗 Link {i+1}/3: Target {i}")
    
    # --- A. GENERATE HONEST OOF PREDS (For Next Link) ---
    honest_feat = None
    if i < 2:
        print(f"      ⚙️ Generating Honest History Features...")
        honest_feat = np.zeros(len(X_tr_curr))
        tscv = TimeSeriesSplit(n_splits=5)
        
        for t_idx, v_idx in tscv.split(X_tr_curr):
            inner_pool_tr = Pool(X_tr_curr[t_idx], y_tr[t_idx, i])
            inner_pool_val = Pool(X_tr_curr[v_idx], y_tr[v_idx, i])
            
            inner_model = CatBoostRegressor(**params_cat)
            inner_model.fit(inner_pool_tr, eval_set=inner_pool_val, early_stopping_rounds=50)
            
            honest_feat[v_idx] = inner_model.predict(X_tr_curr[v_idx])

    # --- B. TRAIN MAIN MODEL ---
    print(f"      Training Main Model...")
    train_pool = Pool(X_tr_curr, y_tr[:, i])
    val_pool = Pool(X_val_curr, y_val[:, i])
    
    model = CatBoostRegressor(**params_cat)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
    
    # Predict
    val_pred = model.predict(X_val_curr)
    test_pred = model.predict(X_test_curr)
    cat_preds[:, i] = test_pred
    
    # --- C. UPDATE FEATURES FOR NEXT LINK ---
    if i < 2:
        print(f"      🔗 Chaining predictions to next target...")
        X_tr_curr = np.hstack([X_tr_curr, honest_feat.reshape(-1, 1)])
        X_val_curr = np.hstack([X_val_curr, val_pred.reshape(-1, 1)])
        X_test_curr = np.hstack([X_test_curr, test_pred.reshape(-1, 1)])

# Save
sub_cat = pd.DataFrame({'id': test_df['id']})
sub_cat[['target_short', 'target_medium', 'target_long']] = cat_preds
for c in target_cols: sub_cat[c] -= sub_cat[c].mean()
sub_cat.to_csv('submission_cat_purist.csv', index=False)
print("🚀 Saved Purist CatBoost.")

🔗 INITIALIZING PURIST CATBOOST CHAIN...

🔗 Link 1/3: Target 0
      ⚙️ Generating Honest History Features...
      Training Main Model...
      🔗 Chaining predictions to next target...

🔗 Link 2/3: Target 1
      ⚙️ Generating Honest History Features...
      Training Main Model...
      🔗 Chaining predictions to next target...

🔗 Link 3/3: Target 2
      Training Main Model...
🚀 Saved Purist CatBoost.


In [13]:
# CELL 15: Titanium Blend (Purist Edition)
import pandas as pd
import numpy as np

print("💎 Blending Purist Models...")

# 1. LOAD PREDICTIONS (Using the Correct Filenames)
files = {
    'XGB': 'submission_xgb_purist.csv',
    'LGB': 'submission_lgbm_purist.csv',
    'CAT': 'submission_cat_purist.csv'
}

preds = {}
for name, path in files.items():
    try:
        preds[name] = pd.read_csv(path)[['target_short', 'target_medium', 'target_long']].values
        print(f"   ✅ Loaded {name}: {path}")
    except FileNotFoundError:
        print(f"   ❌ ERROR: Missing {name}. Did you run the model cell?")
        raise

# 2. BLEND (Classic Safe Weights)
# XGB is usually the most robust in Time Series, so it gets the lead.
w_xgb, w_lgb, w_cat = 0.50, 0.30, 0.20

print(f"   ⚖️ Weights: XGB {w_xgb} | LGB {w_lgb} | CAT {w_cat}")
blend_preds = (preds['XGB'] * w_xgb) + (preds['LGB'] * w_lgb) + (preds['CAT'] * w_cat)

# 3. SAVE
sub = pd.read_csv(files['XGB'])[['id']].copy()
sub['target_short'] = blend_preds[:, 0]
sub['target_medium'] = blend_preds[:, 1]
sub['target_long'] = blend_preds[:, 2]

# Zero-Sum Polish (Optional but good)
for c in sub.columns[1:]:
    sub[c] = sub[c] - sub[c].mean()

sub.to_csv('submission.csv', index=False)
print("🚀 Saved 'submission.csv'. Ready for Private LB.")

💎 Blending Purist Models...
   ✅ Loaded XGB: submission_xgb_purist.csv
   ✅ Loaded LGB: submission_lgbm_purist.csv
   ✅ Loaded CAT: submission_cat_purist.csv
   ⚖️ Weights: XGB 0.5 | LGB 0.3 | CAT 0.2
🚀 Saved 'submission.csv'. Ready for Private LB.
